In [ ]:
# %%
# limit the number of threads on clusters, by Chao, 02/06/2025
import sys, os
os.environ['OMP_NUM_THREADS'] = '10'
os.environ['OPENBLAS_NUM_THREADS'] = '10'
os.environ['MKL_NUM_THREADS'] = '10'
os.environ['VECLIB_MAXIMUM_THREADS'] = '10'
os.environ['NUMEXPR_NUM_THREADS'] = '10'

import pandas as pd
import numpy as np
import utils_plot as utp


In [ ]:
# define the centroid of relative coordinates, must be consistent with the mesh!
lon0, lat0 = -84, 7     # from Christos's email
# print(lon0, lat0)

# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW

# offset in x and y direction, the same as being done to the mesh in 'Kyriakopoulos2016JGR/convert_exodus_to_msh.ipynb'
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m


In [ ]:
# %%
# read in the 3D velocity model
veldir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
vel3dfile = "DeShon2006_3Dmodel.csv"
vel3d = pd.read_csv(veldir + vel3dfile, sep=",")
print(f'X range: {vel3d["lon"].min():.2f} to {vel3d["lon"].max():.2f} degrees')
print(f'Y range: {vel3d["lat"].min():.2f} to {vel3d["lat"].max():.2f} degrees')
print(f'X span: {vel3d["lon"].max() - vel3d["lon"].min():.2f} degrees')
print(f'Y span: {vel3d["lat"].max() - vel3d["lat"].min():.2f} degrees')

# convert to relative locations in meters, and then rotate, then offset, align with the local coordinate system of the mesh
x_rot, y_rot = utp.LL2ckmd(vel3d['lon'], vel3d['lat'], lon0, lat0, rot)
vel3d['x'], vel3d['y'] = x_rot - x0, y_rot - y0   # offset to match the mesh coordinates
vel3d['z'] = vel3d['z'] * -1 * 1e3  # negative depth means downward
vel3d = vel3d[(vel3d['z'] <= 0)].reset_index(drop=True)  # ignore everything above the ground
# print(vel3d.shape)
# print(vel3d.head())
# print(f'X range: {vel3d["x"].min():.2f} to {vel3d["x"].max():.2f} meters')
# print(f'Y range: {vel3d["y"].min():.2f} to {vel3d["y"].max():.2f} meters')
print(f'Z range: {vel3d["z"].min():.2f} to {vel3d["z"].max():.2f} meters')
# print(f'X span: {vel3d["x"].max() - vel3d["x"].min():.2f} meters')
# print(f'Y span: {vel3d["y"].max() - vel3d["y"].min():.2f} meters')


In [ ]:
# %%
import meshio
inname = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(inname)
points = mesh.points  # shape (n_points, 3)
print(mesh.cells)
# Create DataFrame
plate = pd.DataFrame(points, columns=["x", "y", "z"])
plate['lon'], plate['lat'] = utp.ckm2LLd(plate['x'], plate['y'], lon0, lat0, -rot)
# print(plate.head())

print(f'X range: {plate["x"].min() - x0:.2f} to {plate["x"].max() - x0:.2f} meters')
print(f'Y range: {plate["y"].min() - y0:.2f} to {plate["y"].max() - y0:.2f} meters')
print(f'Z range: {plate["z"].min():.2f} to {plate["z"].max():.2f} meters')

In [ ]:
# %%
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Create three-panel figure
fig = plt.figure(figsize=(20, 6), dpi=300)

lon_min, lon_max = vel3d['lon'].min(), vel3d['lon'].max()
lat_min, lat_max = vel3d['lat'].min(), vel3d['lat'].max()
pltcol = 'vs'

# Define the y values you want (in meters, after rotation and shifting)
# y_values = [0, 5e4, -5e4]  # in meters
y_values = [0]  # in meters
# Create x coordinates spanning the range of your data
x_range = np.linspace(vel3d['x'].min(), vel3d['x'].max(), 100)

# Panel 1: Geographical map with Cartopy
ax1 = plt.subplot(1, 3, 1, projection=ccrs.PlateCarree())

# Add map features
ax1.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
ax1.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
ax1.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax1.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')

# Create polygon for lat/lon range
poly_latlon = Polygon([
    [lon_min, lat_min],
    [lon_max, lat_min],
    [lon_max, lat_max],
    [lon_min, lat_max]
], closed=True, edgecolor='red', facecolor='red', alpha=0.3, linewidth=2, transform=ccrs.PlateCarree())

ax1.add_patch(poly_latlon)
ax1.plot(vel3d['lon'], vel3d['lat'], 'k.', markersize=0.5, alpha=0.3, transform=ccrs.PlateCarree(), label='3D model points')
ax1.scatter(plate['lon'], plate['lat'], s=10, color='blue', marker='o', transform=ccrs.PlateCarree())
# ax1.plot(lon0, lat0, 'r*', markersize=15, transform=ccrs.PlateCarree(), label=f'Origin ({lon0}, {lat0})')
lon_origin, lat_origin = utp.ckm2LLd(x0, y0, lon0, lat0, -rot)
ax1.plot(lon_origin, lat_origin, 'r*', markersize=15, transform=ccrs.PlateCarree(), label=f'Origin (0, 0)')

for y_val in y_values:
    # Create arrays of x and y coordinates
    x_coords = x_range
    y_coords = np.full_like(x_range, y_val)
    
    # Reverse the offset (add back x0, y0)
    x_unshifted = x_coords + x0
    y_unshifted = y_coords + y0
    
    # Reverse the rotation and convert back to lon/lat
    # Note: use -rot to reverse the rotation
    lon_line, lat_line = utp.ckm2LLd(x_unshifted, y_unshifted, lon0, lat0, -rot)
    
    # Now you can plot these lines
    ax1.plot(lon_line, lat_line, 'k--', linewidth=1, label=f'y={y_val/1e3:.0f} km')

# Alternative: define line by extending in both directions
lon_center = -85.5
lat_center = 10.3

# Convert to lat/lon offsets
# 1 degree latitude ≈ 111 km
# 1 degree longitude ≈ 111 km * cos(lat)
km_per_deg_lat = 111.0
km_per_deg_lon = 111.0 * np.cos(np.radians(lat_center))

dlat = x_range/1e3 * np.cos(np.radians(rot)) / km_per_deg_lat
dlon = x_range/1e3 * np.sin(np.radians(rot)) / km_per_deg_lon

lat_line = lat_center + dlat
lon_line = lon_center + dlon

# Now convert to x/y and plot as above
x_rot, y_rot = utp.LL2ckmd(lon_line, lat_line, lon0, lat0, rot)
x_line = x_rot - x0
y_line = y_rot - y0

ax1.plot(lon_line, lat_line, 'k--', linewidth=3, label='45° from N')

ax1.set_xlabel('Longitude (degrees)', fontsize=12)
ax1.set_ylabel('Latitude (degrees)', fontsize=12)
ax1.set_title('3D Model Range in Lat/Lon', fontsize=14, fontweight='bold')
ax1.gridlines(draw_labels=True, alpha=0.3)
ax1.legend(loc='upper right')

# Set extent with some padding
pad = 0.5
ax1.set_extent([lon_min-pad, lon_max+pad, lat_min-pad, lat_max+pad], crs=ccrs.PlateCarree())

# Panel 2: Depth vs Latitude
ax2 = plt.subplot(1, 3, 2)

# Plot sampling points in lat-depth space
ax2.scatter(vel3d['lon'], vel3d['z']/1e3, c=vel3d[pltcol], cmap='jet', s=10, alpha=0.5)
ax2.set_xlabel('Longitude (degrees)', fontsize=12)
ax2.set_ylabel('Depth (km)', fontsize=12)
ax2.set_title('3D Model Sampling Points (Lon vs Depth)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([lon_min-pad, lon_max+pad])
ax2.set_ylim([-100, 0])

# Add colorbar
sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=vel3d[pltcol].min(), vmax=vel3d[pltcol].max()))
sm.set_array([])
cbar2 = plt.colorbar(sm, ax=ax2)
cbar2.set_label('Vs (km/s)', fontsize=12)

# Panel 3: Rotated and shifted x/y coordinates
ax3 = plt.subplot(1, 3, 3)

# Transform the CORNERS of the original lat/lon bounding box
corners_lon = [lon_min, lon_max, lon_max, lon_min]
corners_lat = [lat_min, lat_min, lat_max, lat_max]
corners_x_rot, corners_y_rot = utp.LL2ckmd(np.array(corners_lon), np.array(corners_lat), lon0, lat0, rot)
corners_x = (corners_x_rot - x0)/1e3
corners_y = (corners_y_rot - y0)/1e3

# Create polygon for the transformed corners (original extent)
poly_xy = Polygon([
    [corners_x[0], corners_y[0]],
    [corners_x[1], corners_y[1]],
    [corners_x[2], corners_y[2]],
    [corners_x[3], corners_y[3]]
], closed=True, edgecolor='red', facecolor='red', alpha=0.3, linewidth=2, label='Original lat/lon extent')

ax3.add_patch(poly_xy)
ax3.plot(vel3d['x']/1e3, vel3d['y']/1e3, 'k.', markersize=0.5, alpha=0.3, label='3D model points')
ax3.scatter((plate['x'] - x0)/1e3, (plate['y'] - y0)/1e3, s=10, color='blue', marker='o')
ax3.plot(0, 0, 'r*', markersize=15, label='Origin (0, 0)')
for y_val in y_values:
    ax3.axhline(y=y_val/1e3, color='k', linestyle='--', linewidth=1)
ax3.plot(x_line/1e3, y_line/1e3, 'k--', linewidth=3, label='45° from N')
ax3.set_xlabel('X (km)', fontsize=12)
ax3.set_ylabel('Y (km)', fontsize=12)
ax3.set_title(f'3D Model Range in X/Y (rotated {rot}° CCW, shifted)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='best')
ax3.set_aspect('equal', adjustable='box')

# Format axis labels
# ax3.ticklabel_format(style='scientific', axis='both', scilimits=(0,0))

plt.tight_layout()
plt.show()

# Print summary statistics
x_min, x_max = vel3d['x'].min(), vel3d['x'].max()
y_min, y_max = vel3d['y'].min(), vel3d['y'].max()
print(f"\n=== 3D Model Range Summary ===")
print(f"Lat/Lon coordinates:")
print(f"  Lon: {lon_min:.2f}° to {lon_max:.2f}° (span: {lon_max-lon_min:.2f}°)")
print(f"  Lat: {lat_min:.2f}° to {lat_max:.2f}° (span: {lat_max-lat_min:.2f}°)")
print(f"\nRotated/Shifted X/Y coordinates (actual data range):")
print(f"  X: {x_min/1e3:.2f} km to {x_max/1e3:.2f} km (span: {(x_max-x_min)/1e3:.2f} km)")
print(f"  Y: {y_min/1e3:.2f} km to {y_max/1e3:.2f} km (span: {(y_max-y_min)/1e3:.2f} km)")
print(f"\nTransformed corners of original lat/lon extent:")
print(f"  X: {corners_x.min():.2f} km to {corners_x.max():.2f} km")
print(f"  Y: {corners_y.min():.2f} km to {corners_y.max():.2f} km")
print(f"\nDepth range: {vel3d['z'].min()/1e3:.2f} km to {vel3d['z'].max()/1e3:.2f} km")
print(f"\nTotal points: {len(vel3d)}")

In [ ]:
# %%
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# Function to create horizontal slice at a given depth
def plot_horizontal_slice(vel3d, pltcol, depth_km, ax=None, method='linear', resolution=200,
                          add_colorbar=True, add_contour=True, add_contour_labels=True, show_data_points=False,
                          no_interpolation=False, marker='s', marker_size=50, vmin=None, vmax=None,
                          xcol='x', ycol='y', zcol='z', xlim=None, ylim=None, cmap='gist_rainbow_r'):
    """
    Plot horizontal slice of velocity model at a given depth

    Parameters:
    -----------
    vel3d : DataFrame with location and velocity columns
    pltcol : column name to plot (e.g., 'vs', 'vp')
    depth_km : depth in kilometers (positive value, will be converted to negative)
    ax : matplotlib axes object (if None, creates new figure)
    method : interpolation method ('linear', 'nearest', 'cubic')
    resolution : number of grid points in each direction
    add_colorbar : bool, whether to add colorbar (default True)
    add_contour : bool, whether to add contour lines (default True)
    add_contour_labels : bool, whether to add contour labels (default True)
    show_data_points : bool, whether to show data points (default False)
    no_interpolation : bool, if True, plot sparse data directly without interpolation (default False)
    marker : marker style for scatter plot when no_interpolation=True (default 's')
    marker_size : size of markers when no_interpolation=True (default 50)
    vmin : minimum value for colorbar (default None, uses data min)
    vmax : maximum value for colorbar (default None, uses data max)
    xcol : name of x-coordinate column (default 'x', can use 'lon', etc.)
    ycol : name of y-coordinate column (default 'y', can use 'lat', etc.)
    zcol : name of z-coordinate column (default 'z')
    xlim : tuple (xmin, xmax) for x-axis limits (default None, uses data extent)
    ylim : tuple (ymin, ymax) for y-axis limits (default None, uses data extent)

    Returns:
    --------
    plot_obj : contour plot object (or scatter plot object if no_interpolation=True)
    cbar : colorbar object (or None if add_colorbar=False)
    """
    depth_m = -depth_km * 1e3  # convert to negative meters

    # Find data points close to this depth (within ±5 km)
    tolerance = 5e3  # 5 km tolerance
    mask = np.abs(vel3d[zcol] - depth_m) < tolerance
    data_slice = vel3d[mask]

    if len(data_slice) == 0:
        print(f"No data found at depth {depth_km} km")
        return None, None

    # Create axes if not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
        standalone = True
    else:
        standalone = False

    # Extract values for plotting
    values = data_slice[pltcol].values
    
    # Set colorbar limits - use global min/max if not specified
    if vmin is None:
        vmin = vel3d[pltcol].min()
    if vmax is None:
        vmax = vel3d[pltcol].max()
    
    # cmap = 'coolwarm'
    # cmap = 'gist_rainbow_r'
    # # cmap = 'jet'

    # Determine coordinate units for labels and whether we're using geographic coordinates
    # If using lon/lat, don't divide by 1e3, otherwise convert to km
    is_geographic = False
    if xcol.lower() in ['lon', 'longitude']:
        x_scale = 1
        x_unit = 'degrees'
        is_geographic = True
    else:
        x_scale = 1e3
        x_unit = 'km'
    
    if ycol.lower() in ['lat', 'latitude']:
        y_scale = 1
        y_unit = 'degrees'
        is_geographic = True
    else:
        y_scale = 1e3
        y_unit = 'km'

    # Get data extent for tight limits (use manual limits if provided)
    if xlim is None:
        x_data_min, x_data_max = data_slice[xcol].min() / x_scale, data_slice[xcol].max() / x_scale
    else:
        x_data_min, x_data_max = xlim
        
    if ylim is None:
        y_data_min, y_data_max = data_slice[ycol].min() / y_scale, data_slice[ycol].max() / y_scale
    else:
        y_data_min, y_data_max = ylim

    # Plot based on interpolation mode
    if no_interpolation:
        # Plot sparse data directly without interpolation
        scatter = ax.scatter(data_slice[xcol]/x_scale, data_slice[ycol]/y_scale,
                           c=values, s=marker_size, cmap=cmap,
                           vmin=vmin, vmax=vmax, marker=marker,
                           edgecolors='none', linewidths=0.5, alpha=0.8)
        plot_obj = scatter  # Return scatter object for consistency

        # Add colorbar if requested
        cbar = None
        if add_colorbar:
            cbar = plt.colorbar(scatter, ax=ax, shrink=0.8, pad=0.02)
            cbar.set_label(f'{pltcol} (km/s)', fontsize=12)
            # Add this line for better ticks:
            cbar.set_ticks(np.linspace(vmin, vmax, 6))  # 6 nice evenly-spaced ticks
    else:
        # Original interpolation method
        # Create grid
        x_min, x_max = vel3d[xcol].min(), vel3d[xcol].max()
        y_min, y_max = vel3d[ycol].min(), vel3d[ycol].max()
        xi = np.linspace(x_min, x_max, resolution)
        yi = np.linspace(y_min, y_max, resolution)
        Xi, Yi = np.meshgrid(xi, yi)

        # Interpolate values
        points = np.column_stack((data_slice[xcol], data_slice[ycol]))
        Zi = griddata(points, values, (Xi, Yi), method=method)

        # Plot with specified vmin/vmax
        levels = np.linspace(vmin, vmax, 256)
        plot_obj = ax.contourf(Xi/x_scale, Yi/y_scale, Zi, levels=levels, cmap=cmap, vmin=vmin, vmax=vmax, extend='both')
        
        # Add contour lines if requested
        if add_contour:
            contour_levels = np.linspace(vmin, vmax, 10)
            contour = ax.contour(Xi/x_scale, Yi/y_scale, Zi, levels=contour_levels, colors='k', linewidths=0.5, alpha=0.3)

            if add_contour_labels:
                ax.clabel(contour, inline=True, fontsize=8)

        # Add data points if requested
        if show_data_points:
            ax.scatter(data_slice[xcol]/x_scale, data_slice[ycol]/y_scale, c='white', s=1, alpha=0.3, label='Data points')

        # Add colorbar if requested
        cbar = None
        if add_colorbar:
            cbar = plt.colorbar(plot_obj, ax=ax, shrink=0.8, pad=0.02)
            cbar.set_label(f'{pltcol} (km/s)', fontsize=12)
            # Add this line for better ticks:
            cbar.set_ticks(np.linspace(vmin, vmax, 6))  # 6 nice evenly-spaced ticks

    ax.set_xlabel(f'{xcol.upper()} ({x_unit})', fontsize=12)
    ax.set_ylabel(f'{ycol.upper()} ({y_unit})', fontsize=12)
    interp_str = "no interpolation" if no_interpolation else f"interpolated ({method})"
    ax.set_title(f'Horizontal Slice of {pltcol} at Depth = {depth_km} km ({interp_str})\n({len(data_slice)} points within ±5 km)',
                 fontsize=14, fontweight='bold')
    
    # Set tight axis limits based on actual data extent or manual specification
    ax.set_xlim(x_data_min, x_data_max)
    ax.set_ylim(y_data_min, y_data_max)
    
    # Set aspect ratio AFTER setting limits
    # Use 'box' so the plot box adjusts to maintain aspect, not the data limits
    if is_geographic:
        # For geographic coordinates, use the mean latitude to calculate proper aspect ratio
        # cos(lat) accounts for longitude compression at higher latitudes
        if ycol.lower() in ['lat', 'latitude']:
            mean_lat = data_slice[ycol].mean()
            aspect_ratio = 1.0 / np.cos(np.radians(mean_lat))
            ax.set_aspect(aspect_ratio, adjustable='box')
        else:
            ax.set_aspect('equal', adjustable='box')
    else:
        # For x/y in km, use equal aspect
        ax.set_aspect('equal', adjustable='box')
    
    ax.grid(True, alpha=0.3)

    # Only show if standalone
    if standalone:
        plt.tight_layout()
        plt.show()

    print(f"{pltcol} range at depth {depth_km} km: {values.min():.2f} - {values.max():.2f} km/s")
    print(f"Colorbar range: {vmin:.2f} - {vmax:.2f} km/s")
    print(f"Number of data points: {len(data_slice)}")

    return plot_obj, cbar


# Function to create vertical slice at a given x or y position
def plot_vertical_slice(vel3d, pltcol, position_km, direction='x', ax=None, method='linear', resolution=200,
                        add_colorbar=True, add_contour=True, add_contour_labels=True, show_data_points=False,
                        no_interpolation=False, marker='s', marker_size=50, vmin=None, vmax=None,
                        xcol='x', ycol='y', zcol='z', xlim=None, ylim=None, cmap='gist_rainbow_r'):
    """
    Plot vertical slice of velocity model at a given x or y position
    
    Parameters:
    -----------
    vel3d : DataFrame with location and velocity columns
    pltcol : column name to plot (e.g., 'vs', 'vp')
    position_km : position in kilometers along x or y axis
    direction : 'x' for slice perpendicular to x-axis (varying y,z), 
                'y' for slice perpendicular to y-axis (varying x,z)
    ax : matplotlib axes object (if None, creates new figure)
    method : interpolation method ('linear', 'nearest', 'cubic')
    resolution : number of grid points in each direction
    add_colorbar : bool, whether to add colorbar (default True)
    add_contour : bool, whether to add contour lines (default True)
    add_contour_labels : bool, whether to add contour labels (default True)
    show_data_points : bool, whether to show data points (default False)
    no_interpolation : bool, if True, plot sparse data directly without interpolation (default False)
    marker : marker style for scatter plot when no_interpolation=True (default 's')
    marker_size : size of markers when no_interpolation=True (default 50)
    vmin : minimum value for colorbar (default None, uses data min)
    vmax : maximum value for colorbar (default None, uses data max)
    xcol : name of x-coordinate column (default 'x', can use 'lon', etc.)
    ycol : name of y-coordinate column (default 'y', can use 'lat', etc.)
    zcol : name of z-coordinate column (default 'z')
    xlim : tuple (xmin, xmax) for horizontal axis limits (default None, uses data extent)
    ylim : tuple (ymin, ymax) for depth axis limits (default None, uses data extent)
    
    Returns:
    --------
    plot_obj : contour plot object (or scatter plot object if no_interpolation=True)
    cbar : colorbar object (or None if add_colorbar=False)
    """
    position_m = position_km * 1e3
    
    # Find data points close to this position (within ±10 km)
    tolerance = 10e3  # 10 km tolerance
    
    # Determine which coordinate column to use
    if direction == 'x':
        slice_col = xcol
        coord_col = ycol
        coord_label_base = ycol.upper()
    else:  # direction == 'y'
        slice_col = ycol
        coord_col = xcol
        coord_label_base = xcol.upper()
    
    # Determine coordinate units for labels
    if coord_col.lower() in ['lon', 'longitude', 'lat', 'latitude']:
        coord_scale = 1
        coord_unit = 'degrees'
    else:
        coord_scale = 1e3
        coord_unit = 'km'
    
    mask = np.abs(vel3d[slice_col] - position_m) < tolerance
    data_slice = vel3d[mask]
    
    if len(data_slice) == 0:
        print(f"No data found at {slice_col} = {position_km} km")
        return None, None
    
    coord_data = data_slice[coord_col]
    coord_min, coord_max = vel3d[coord_col].min(), vel3d[coord_col].max()
    title_pos = f'{slice_col.upper()} = {position_km} km'
    
    # Extract values for plotting
    values = data_slice[pltcol].values
    
    # Set colorbar limits - use global min/max if not specified
    if vmin is None:
        vmin = vel3d[pltcol].min()
    if vmax is None:
        vmax = vel3d[pltcol].max()
    
    # cmap = 'coolwarm'
    # cmap = 'gist_rainbow_r'
    # # cmap = 'jet'
    
    # Create axes if not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
        standalone = True
    else:
        standalone = False
    
    # Get data extent for tight limits (use manual limits if provided)
    if xlim is None:
        coord_data_min = data_slice[coord_col].min() / coord_scale
        coord_data_max = data_slice[coord_col].max() / coord_scale
    else:
        coord_data_min, coord_data_max = xlim
        
    if ylim is None:
        z_data_min = data_slice[zcol].min() / 1e3
        z_data_max = data_slice[zcol].max() / 1e3
    else:
        z_data_min, z_data_max = ylim
    
    if no_interpolation:
        # Plot sparse data directly without interpolation
        scatter = ax.scatter(coord_data/coord_scale, data_slice[zcol]/1e3,
                           c=values, s=marker_size, cmap=cmap,
                           vmin=vmin, vmax=vmax, marker=marker,
                           edgecolors='none', linewidths=0.5, alpha=0.8)
        plot_obj = scatter
        
        # Add colorbar if requested
        cbar = None
        if add_colorbar:
            cbar = plt.colorbar(scatter, ax=ax, shrink=0.8, pad=0.02)
            cbar.set_label(f'{pltcol} (km/s)', fontsize=12)
            # Add this line for better ticks:
            cbar.set_ticks(np.linspace(vmin, vmax, 6))  # 6 nice evenly-spaced ticks
        
    else:
        # Original interpolation method
        # Create grid
        z_min, z_max = vel3d[zcol].min(), vel3d[zcol].max()
        ci = np.linspace(coord_min, coord_max, resolution)
        zi = np.linspace(z_min, z_max, resolution)
        Ci, Zi = np.meshgrid(ci, zi)
        
        # Interpolate values
        points = np.column_stack((coord_data, data_slice[zcol]))
        Vi = griddata(points, values, (Ci, Zi), method=method)
        
        levels = np.linspace(vmin, vmax, 256)
        plot_obj = ax.contourf(Ci/coord_scale, Zi/1e3, Vi, levels=levels, cmap=cmap, vmin=vmin, vmax=vmax, extend='both')
        
        # Add contour lines if requested
        if add_contour:
            contour_levels = np.linspace(vmin, vmax, 10)
            contour = ax.contour(Ci/coord_scale, Zi/1e3, Vi, levels=contour_levels, colors='k', linewidths=0.5, alpha=0.3)
            
            if add_contour_labels:
                ax.clabel(contour, inline=True, fontsize=8)
        
        # Add data points if requested
        if show_data_points:
            ax.scatter(coord_data/coord_scale, data_slice[zcol]/1e3, c='white', s=1, alpha=0.5, label='Data points')
        
        # Add colorbar if requested
        cbar = None
        if add_colorbar:
            cbar = plt.colorbar(plot_obj, ax=ax, shrink=0.8, pad=0.02)
            cbar.set_label(f'{pltcol} (km/s)', fontsize=12)
            # Add this line for better ticks:
            cbar.set_ticks(np.linspace(vmin, vmax, 6))  # 6 nice evenly-spaced ticks
    
    ax.set_xlabel(f'{coord_label_base} ({coord_unit})', fontsize=12)
    ax.set_ylabel('Depth (km)', fontsize=12)
    interp_str = "no interpolation" if no_interpolation else f"interpolated ({method})"
    ax.set_title(f'Vertical Slice of {pltcol} at {title_pos} ({interp_str})\n({len(data_slice)} points within ±10 km)', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Set axis limits based on actual data extent or manual specification
    ax.set_xlim(coord_data_min, coord_data_max)
    ax.set_ylim(z_data_min, z_data_max)
    
    # Don't force aspect ratio for vertical slices - let it be natural
    # The plot box will adjust to show the data extent without distortion
    ax.set_aspect('equal', adjustable='box')
    
    # Only show if standalone
    if standalone:
        plt.tight_layout()
        plt.show()
    
    print(f"{pltcol} range in this slice: {values.min():.2f} - {values.max():.2f} km/s")
    print(f"Colorbar range: {vmin:.2f} - {vmax:.2f} km/s")
    print(f"Number of data points: {len(data_slice)}")
    
    return plot_obj, cbar

In [ ]:
# # %% 
# # Plot the original velocity model at different depths, to compare with DeShon et al 2006 Figure 7
# fig, axes = plt.subplots(4, 1, figsize=(5, 20), dpi=300)
# axes = axes.flatten()  # Flatten to 1D array for easier indexing

# pltcol = 'vp'
# vmin, vmax = 4, 9

# print("\nPlotting with Lon/Lat coordinates...")
# print("\nCreating non-interpolated version...")

# # Define depths for slices (you can customize this)
# depths = [9, 20, 35, 50]  # km

# # Plot each slice at a certain depth in a subplot with consistent colorbar
# for i, depth in enumerate(depths):
#     plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=axes[i], vmin=vmin, vmax=vmax,
#                         no_interpolation=True, add_colorbar=True, marker_size=20,
#                         xcol='lon', ycol='lat', zcol='z')
    
# # Add overall title
# fig.suptitle('Horizontal Slices of Vp at Different Depths without interpolation', 
#              fontsize=16, fontweight='bold', y=0.995)  

In [ ]:
# # Plot the rotated original velocity model at different depths, to compare with DeShon et al 2006 Figure 7
# fig, axes = plt.subplots(4, 1, figsize=(5, 20), dpi=300)
# axes = axes.flatten()  # Flatten to 1D array for easier indexing

# pltcol = 'vp'
# vmin, vmax = 4, 9

# print("\nPlotting with X/Y coordinates...")
# print("\nCreating non-interpolated version...")

# # Define depths for slices (you can customize this)
# depths = [9, 20, 35, 50]  # km

# # Plot each slice at a certain depth in a subplot with consistent colorbar
# for i, depth in enumerate(depths):
#     plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=axes[i], vmin=vmin, vmax=vmax,
#                         no_interpolation=True, add_colorbar=True, marker='D', marker_size=20,
#                         xcol='x', ycol='y', zcol='z')
    
# # Add overall title
# fig.suptitle('Horizontal Slices of Vp at Different Depths without interpolation', 
#              fontsize=16, fontweight='bold', y=0.995)    

In [ ]:
# # Plot the rotated & interpolated original velocity model at different depths, to compare with DeShon et al 2006 Figure 7
# fig, axes = plt.subplots(4, 1, figsize=(5, 20), dpi=300)
# axes = axes.flatten()  # Flatten to 1D array for easier indexing

# pltcol = 'vp'
# vmin, vmax = 4, 9

# print("\nPlotting with X/Y coordinates...")
# print("\nCreating interpolated version...")

# # Define depths for slices (you can customize this)
# depths = [9, 20, 35, 50]  # km

# # Plot each slice at a certain depth in a subplot with consistent colorbar
# for i, depth in enumerate(depths):
#     plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=axes[i], resolution=100, 
#                           vmin=vmin, vmax=vmax, add_colorbar=True, add_contour_labels=False,
#                           xcol='x', ycol='y', zcol='z')
    
# # Add overall title
# fig.suptitle('Horizontal Slices of Vp at Different Depths with interpolation', 
#              fontsize=16, fontweight='bold', y=0.995)    

In [ ]:
# # %%
# # Example: Side-by-side comparison of interpolated vs non-interpolated
# fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=300)
# axes = axes.flatten()  # Flatten to 1D array for easier indexing

# # %%
# # Example: Using lon/lat coordinates instead of x/y
# # This is useful when you want to plot in geographic coordinates

# pltcol = 'vp'
# vmin, vmax = 4, 9
# depth_km = 50

# print("\nPlotting with Lon/Lat coordinates...")

# # Left: Without interpolation (sparse data)
# print("\nCreating non-interpolated version...")
# plot_horizontal_slice(vel3d, pltcol, depth_km=depth_km, ax=axes[0], vmin=vmin, vmax=vmax,
#                       no_interpolation=True, add_colorbar=True, marker_size=70,
#                       xcol='lon', ycol='lat', zcol='z')

# # Right: With interpolation (default)
# print("Creating interpolated version...")
# plot_horizontal_slice(vel3d, pltcol, depth_km=depth_km, ax=axes[1], resolution=100, vmin=4, vmax=9,
#                       add_colorbar=True, add_contour_labels=False,
#                       xcol='lon', ycol='lat', zcol='z')


# print("\nPlotting with X/Y coordinates...")
# # Left: Without interpolation (sparse data)
# plot_horizontal_slice(vel3d, pltcol, depth_km=depth_km, ax=axes[2], vmin=4, vmax=9,
#                       no_interpolation=True, add_colorbar=True, marker='D', marker_size=40,
#                       xcol='x', ycol='y', zcol='z')  # Explicitly specify x/y/z

# # Right: With interpolation (default)
# plot_horizontal_slice(vel3d, pltcol, depth_km=depth_km, ax=axes[3], resolution=100, vmin=4, vmax=9,
#                       add_colorbar=True, add_contour_labels=False,
#                       xcol='x', ycol='y', zcol='z')  # Explicitly specify x/y/z


# fig.suptitle('Comparison: X/Y (km) vs Lon/Lat (degrees) Coordinates ', 
#              fontsize=16, fontweight='bold', y=0.98)
# fig.suptitle('Comparison: Interpolated vs Non-Interpolated Visualization', 
#              fontsize=16, fontweight='bold', y=0.98)

# plt.tight_layout()
# plt.show()

In [ ]:
# Plot the rotated & interpolated original velocity model at different depths, to compare with DeShon et al 2006 Figure 7
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=300)
axes = axes.flatten()  # Flatten to 1D array for easier indexing

pltcol = 'vs'
# pltcol = 'vp'

if pltcol == 'vp':
    vmin_global = 4
    vmax_global = 9
elif pltcol == 'vs':
    # Get global min/max for consistent colorbar
    vmin_global = vel3d[pltcol].min()
    vmax_global = vel3d[pltcol].max()
    print(f"Global Vs range: {vmin_global:.2f} - {vmax_global:.2f} km/s\n")
    print(f"Global Vs range: {vmin_global:.2f} - {vmax_global:.2f} km/s\n")
    vmin_global = 2.5
    vmax_global = 6

print("\nPlotting with X/Y coordinates...")
print("\nCreating interpolated version...")

# Define depths for slices (you can customize this)
depths = [9, 20, 35, 50]  # km

# Plot each slice at a certain depth in a subplot with consistent colorbar
for i, depth in enumerate(depths):
    # plot_obj, _ = plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=axes[i], resolution=250, 
    #                       vmin=vmin_global, vmax=vmax_global, 
    #                       no_interpolation=True, marker='D', marker_size=25,
    #                       add_colorbar=True, add_contour_labels=False,
    #                       xcol='x', ycol='y', zcol='z')

    plot_obj, _ = plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=axes[i], resolution=250, 
                          vmin=vmin_global, vmax=vmax_global, 
                          add_colorbar=True, add_contour_labels=False,
                          xcol='x', ycol='y', zcol='z')

    for y_val in y_values:
        axes[i].axhline(y=y_val/1e3, color='k', linestyle='--', linewidth=1)
    axes[i].plot(x_line/1e3, y_line/1e3, 'k--', linewidth=3, label='45° from N')

# Add overall title
fig.suptitle('Horizontal Slices of Vs at Different Depths', 
             fontsize=16, fontweight='bold', y=0.995)    

In [ ]:
# %% 
# Example: Vertical slice at varying x (longitude vs depth), at the SAME cross-secion as Figure 3 of DeShon & Schwartz 2004 GRL 
print("=== Vertical Slice: Longitude vs Depth at Y=0 km ===\n")

fig, axes = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
# axes = axes.flatten()  # Flatten to 1D array for easier indexing

pltcol = 'vp'
vmin, vmax = 4, 9

y_pos_km = np.round(y_line.mean())/1e3

# Slice at y=0 km, showing longitude (x-axis) vs depth (y-axis)
plot_obj, _ = plot_vertical_slice(vel3d, pltcol, ax=axes, position_km=y_pos_km, direction='y',  # slice at y=0, varying x
                                vmin=vmin, vmax=vmax, resolution=250, add_contour=False,
                                add_colorbar=True, add_contour_labels=False,
                                xcol='x', ycol='y', zcol='z', xlim=[-100, 80], ylim=[-90, 0])  # x-axis displays as longitude

In [ ]:
# Example: Vertical slice at varying x (longitude vs depth), at the SAME cross-secion as Figure 3 of DeShon & Schwartz 2004 GRL 
print("=== Vertical Slice: Longitude vs Depth at Y=0 km ===\n")

fig, axes = plt.subplots(1, 1, figsize=(15, 5), dpi=300)
# axes = axes.flatten()  # Flatten to 1D array for easier indexing

pltcol = 'vs'
vmin, vmax = 2.8, 5.8

y_pos_km = 0

# Slice at y=0 km, showing longitude (x-axis) vs depth (y-axis)
plot_obj, _ = plot_vertical_slice(vel3d, pltcol, ax=axes, position_km=y_pos_km, direction='y',  # slice at y=0, varying x
                                vmin=vmin, vmax=vmax, resolution=250, add_contour=False,
                                add_colorbar=True, add_contour_labels=False, cmap='jet',
                                xcol='x', ycol='y', zcol='z', xlim=[-160, 140], ylim=[-90, 0])  # x-axis displays as longitude

In [ ]:
# %%
# ========================================
# PyVista Visualization for Mesh-Projected Models
# ========================================
import pyvista as pv

def load_xdmf_as_pyvista(filepath):
    """
    Load .xdmf file and convert to PyVista UnstructuredGrid
    
    Parameters:
    -----------
    filepath : str
        Path to the .xdmf file
        
    Returns:
    --------
    grid : pv.UnstructuredGrid
        PyVista grid with velocity data
    """
    import os
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    
    m = meshio.read(filepath)

    # Map meshio to pyvista CellType
    cell_type_map = {
        "triangle": pv.CellType.TRIANGLE,
        "tetra": pv.CellType.TETRA,
        "hexahedron": pv.CellType.HEXAHEDRON,
        "quad": pv.CellType.QUAD,
        "wedge": pv.CellType.WEDGE,
    }

    cell_type = next(iter(m.cells_dict))
    if cell_type not in cell_type_map:
        raise ValueError(f"Unsupported cell type: {cell_type}")

    cells = m.cells_dict[cell_type]
    points = m.points

    grid = pv.UnstructuredGrid({cell_type_map[cell_type]: cells}, points)

    # Add all point data fields
    for field_name in m.point_data.keys():
        grid[field_name] = m.point_data[field_name]

    return grid


def plot_horizontal_slice_pyvista(grid, field_name, depth_m, ax=None, vmin=None, vmax=None,
                                   cmap='gist_rainbow_r', add_colorbar=True, levels=256,
                                   xlim=None, ylim=None):
    """
    Plot horizontal slice of PyVista grid at a given depth using matplotlib
    
    Parameters:
    -----------
    grid : pv.UnstructuredGrid
        PyVista grid with velocity data
    field_name : str
        Name of the field to plot (e.g., 'vs', 'vp')
    depth_m : float
        Depth in meters (negative value)
    ax : matplotlib axes
        Axes to plot on (if None, creates new figure)
    vmin, vmax : float
        Color limits
    cmap : str
        Colormap name
    add_colorbar : bool
        Whether to add colorbar
    levels : int
        Number of contour levels
    xlim : tuple (xmin, xmax)
        X-axis limits in km (default None, uses data extent)
    ylim : tuple (ymin, ymax)
        Y-axis limits in km (default None, uses data extent)
        
    Returns:
    --------
    CS : contourf object
    cbar : colorbar object or None
    """
    import matplotlib.pyplot as plt
    import matplotlib.tri as mtri
    
    # Slice the grid horizontally
    normal = [0., 0., 1.]
    origin = [0., 0., depth_m]
    slice_mesh = grid.slice(normal=normal, origin=origin)
    
    if len(slice_mesh.points) == 0:
        print(f"No data found at depth {depth_m/1e3:.1f} km")
        return None, None
    
    # Create axes if not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
        standalone = True
    else:
        standalone = False
    
    # Extract coordinates and field values
    x = slice_mesh.points[:, 0] / 1e3  # Convert to km
    y = slice_mesh.points[:, 1] / 1e3
    
    # Get field values
    if field_name not in slice_mesh.array_names:
        print(f"Available fields: {slice_mesh.array_names}")
        raise ValueError(f"Field '{field_name}' not found in grid")
    
    if field_name == 'shear modulus':
        unit = 'GPa'
    elif field_name == 'shear velocity Vs':
        unit = 'km/s'
    elif field_name == 'density':    
        unit = 'kg m/s^2'

    values = slice_mesh[field_name]
    
    # Set color limits
    if vmin is None:
        vmin = values.min()
    if vmax is None:
        vmax = values.max()
    
    # Create triangulation
    triang = mtri.Triangulation(x, y)
    
    # Plot filled contours
    contour_levels = np.linspace(vmin, vmax, levels)
    CS = ax.tricontourf(triang, values, levels=contour_levels, cmap=cmap, 
                        vmin=vmin, vmax=vmax, extend='both')
    
    # Add colorbar
    cbar = None
    if add_colorbar:
        cbar = plt.colorbar(CS, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label(f'{field_name} ({unit})', fontsize=12)
        cbar.set_ticks(np.linspace(vmin, vmax, 6))
    
    ax.set_xlabel('X (km)', fontsize=12)
    ax.set_ylabel('Y (km)', fontsize=12)
    ax.set_title(f'Horizontal Slice of {field_name} at Depth = {-depth_m/1e3:.1f} km (Mesh-Projected)\n({len(values)} points)',
                 fontsize=14, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    
    # Set axis limits if provided
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    
    if standalone:
        plt.tight_layout()
        plt.show()
    
    print(f"{field_name} range: {values.min():.2f} - {values.max():.2f} {unit}")
    
    return CS, cbar


def plot_vertical_slice_pyvista(grid, field_name, position_m, direction='y', ax=None,
                                vmin=None, vmax=None, cmap='gist_rainbow_r',
                                add_colorbar=True, levels=256):
    """
    Plot vertical slice of PyVista grid
    
    Parameters:
    -----------
    grid : pv.UnstructuredGrid
        PyVista grid with velocity data
    field_name : str
        Name of the field to plot
    position_m : float
        Position in meters along the slicing direction
    direction : str
        'x' or 'y' - direction perpendicular to slice
    ax : matplotlib axes
        Axes to plot on
    vmin, vmax : float
        Color limits
    cmap : str
        Colormap name
    add_colorbar : bool
        Whether to add colorbar
    levels : int
        Number of contour levels
        
    Returns:
    --------
    CS : contourf object
    cbar : colorbar object or None
    """
    import matplotlib.pyplot as plt
    import matplotlib.tri as mtri
    
    # Define slice normal and origin
    if direction == 'y':
        normal = [0., 1., 0.]
        origin = [0., position_m, 0.]
        coord_label = 'X'
        coord_idx = 0
    else:  # direction == 'x'
        normal = [1., 0., 0.]
        origin = [position_m, 0., 0.]
        coord_label = 'Y'
        coord_idx = 1
    
    # Slice the grid
    slice_mesh = grid.slice(normal=normal, origin=origin)
    
    if len(slice_mesh.points) == 0:
        print(f"No data found at {direction} = {position_m/1e3:.1f} km")
        return None, None
    
    # Create axes if not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
        standalone = True
    else:
        standalone = False
    
    # Extract coordinates
    coord = slice_mesh.points[:, coord_idx] / 1e3  # km
    z = slice_mesh.points[:, 2] / 1e3  # km
    
    # Get field values
    if field_name not in slice_mesh.array_names:
        raise ValueError(f"Field '{field_name}' not found in grid")
    
    if field_name == 'shear modulus':
        unit = 'GPa'
    elif field_name == 'shear velocity Vs':
        unit = 'km/s'
    elif field_name == 'density':    
        unit = 'kg m/s^2'
    
    values = slice_mesh[field_name]
    
    # Set color limits
    if vmin is None:
        vmin = values.min()
    if vmax is None:
        vmax = values.max()
    
    # Create triangulation
    triang = mtri.Triangulation(coord, z)
    
    # Plot filled contours
    contour_levels = np.linspace(vmin, vmax, levels)
    CS = ax.tricontourf(triang, values, levels=contour_levels, cmap=cmap,
                        vmin=vmin, vmax=vmax, extend='both')
    
    # Add colorbar
    cbar = None
    if add_colorbar:
        cbar = plt.colorbar(CS, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label(f'{field_name} ({unit})', fontsize=12)
        cbar.set_ticks(np.linspace(vmin, vmax, 6))
    
    ax.set_xlabel(f'{coord_label} (km)', fontsize=12)
    ax.set_ylabel('Depth (km)', fontsize=12)
    ax.set_title(f'Vertical Slice of {field_name} at {direction.upper()} = {position_m/1e3:.1f} km (Mesh-Projected)\n({len(values)} points)',
                 fontsize=14, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    
    if standalone:
        plt.tight_layout()
        plt.show()
    
    print(f"{field_name} range: {values.min():.2f} - {values.max():.2f} {unit}")
    
    return CS, cbar

print("PyVista functions loaded successfully!")

In [ ]:
# %%
# Load the mesh-projected velocity model (.xdmf file)

# original model
xdmf_dir = "/home/staff/chao/SSEinv/Nicoya/rst_locking/"
# vs_file = "vs_true_nicoyaCK3_DeShon3D_ref_1.xdmf"
# mu_file = "mu_true_nicoyaCK3_DeShon3D_ref_1.xdmf"
# pho_file = "pho_true_nicoyaCK3_DeShon3D_ref_1.xdmf"

vs_file = "vs_true_nicoyaCK3_DeShon3D_ref_4.xdmf"
mu_file = "mu_true_nicoyaCK3_DeShon3D_ref_4.xdmf"
pho_file = "pho_true_nicoyaCK3_DeShon3D_ref_4.xdmf"

# xdmf_dir = "/home/staff/chao/SSEinv/Nicoya/syn_slip/"

# # probably okay to use, inspected with TWB  
# vs_file = "vs_true_nicoyaCK3_dense2_sm_DeShon3D_ref_1.xdmf"
# mu_file = "mu_true_nicoyaCK3_dense2_sm_DeShon3D_ref_1.xdmf"
# pho_file = "pho_true_nicoyaCK3_dense2_sm_DeShon3D_ref_1.xdmf"

# #latest model, denser, v4
# vs_file = "vs_true_nicoyaCK3_dense4_DeShon3D_ref_1.xdmf"
# mu_file = "mu_true_nicoyaCK3_dense4_DeShon3D_ref_1.xdmf"
# pho_file = "pho_true_nicoyaCK3_dense4_DeShon3D_ref_1.xdmf"

# #latest model, same geometry as v4, but slightly coarser
# vs_file = "vs_true_nicoyaCK3_dense5_DeShon3D_ref_1.xdmf"
# mu_file = "mu_true_nicoyaCK3_dense5_DeShon3D_ref_1.xdmf"
# pho_file = "pho_true_nicoyaCK3_dense5_DeShon3D_ref_1.xdmf"

# #latest model, same geometry as v4, even coarser
# vs_file = "vs_true_nicoyaCK3_dense6_DeShon3D_ref_1.xdmf"
# mu_file = "mu_true_nicoyaCK3_dense6_DeShon3D_ref_1.xdmf"
# pho_file = "pho_true_nicoyaCK3_dense6_DeShon3D_ref_1.xdmf"

# vs_file = "vs_true_nicoyaCK3_dense7_sm_DeShon3D_ref_1.xdmf"
# mu_file = "mu_true_nicoyaCK3_dense7_sm_DeShon3D_ref_1.xdmf"
# pho_file = "pho_true_nicoyaCK3_dense7_sm_DeShon3D_ref_1.xdmf"

print(f"Loading {vs_file}...")
vs_grid = load_xdmf_as_pyvista(xdmf_dir + vs_file)

print(f"\nGrid info:")
print(f"  Number of points: {vs_grid.n_points}")
print(f"  Number of cells: {vs_grid.n_cells}")
print(f"  Bounds (m): {vs_grid.bounds}")
print(f"  Available fields: {vs_grid.array_names}")

# Check the field name - it might be 'vs', 'Vs', 'velocity', etc.
field_name = vs_grid.array_names[0] if len(vs_grid.array_names) > 0 else 'vs'
print(f"\nUsing field: '{field_name}'")

In [ ]:
# %%
# ========================================
# COMPARISON: Original CSV Model vs Mesh-Projected Model (Horizontal Slices)
# ========================================

fig, axes = plt.subplots(2, 4, figsize=(20, 9), dpi=300)

pltcol = 'vs'
vmin_global = 2.5
vmax_global = 6

y_values = [-50, -25, 0, 25, 50]  # km
depths = [9, 20, 35, 50]  # km

print("=" * 60)
print("HORIZONTAL SLICE COMPARISON")
print("=" * 60)

for i, depth in enumerate(depths):
    print(f"\nDepth: {depth} km")
    print("-" * 40)
    
    # Top row: Original CSV model
    print("  Plotting original CSV model...")
    plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=axes[0, i], resolution=250,
                         vmin=vmin_global, vmax=vmax_global, 
                         xlim=[-350, 350], ylim=[-300, 350],
                         add_colorbar=True, add_contour=False,
                         no_interpolation=True, marker='D', marker_size=40,
                         xcol='x', ycol='y', zcol='z')
    
    for y_val in y_values:
        axes[0, i].axhline(y=y_val/1e3, color='k', linestyle='--', linewidth=1)
    # axes[0, i].plot(x_line/1e3, y_line/1e3, 'k--', linewidth=3, label='45° from N')
    axes[0, i].set_title(f'Original CSV\nDepth = {depth} km', fontsize=12, fontweight='bold')
    
    # Bottom row: Mesh-projected model
    print("  Plotting mesh-projected model...")
    plot_horizontal_slice_pyvista(vs_grid, field_name, depth_m=-depth*1e3, ax=axes[1, i],
                                  vmin=vmin_global, vmax=vmax_global, 
                                  xlim=[-350, 350], ylim=[-300, 350],
                                  # xlim=[-500, 500], ylim=[-500, 500],
                                #   xlim=[-1000, 1000], ylim=[-1000, 1000],
                                  cmap='gist_rainbow_r', add_colorbar=True, levels=256)

    # from smooth_pyvista_viz import plot_horizontal_slice_pyvista_smooth
    # plot_horizontal_slice_pyvista_smooth(vs_grid, field_name, depth_m=-depth*1e3, ax=axes[1, i],
    #                               vmin=vmin_global, vmax=vmax_global, 
    #                               xlim=[-350, 350], ylim=[-300, 350],
    #                             #   xlim=[-500, 500], ylim=[-500, 500],
    #                             #   xlim=[-1000, 1000], ylim=[-1000, 1000],
    #                               cmap='gist_rainbow_r', add_colorbar=True, resolution=500)

    axes[1, i].set_title(f'Mesh-Projected\nDepth = {depth} km', fontsize=12, fontweight='bold')

fig.suptitle('Comparison: Original vs Mesh-Projected Velocity Model (Horizontal Slices)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# %%
pltcol = 'vs'

vmin_global = 2.5
vmax_global = 6

# y_pos_km = 0  # Position for vertical slice
# y_pos_km = np.round(y_line.mean())/1e3
# y_values = [-50, -25, 0, 25, 50]
# y_values = [-50, -30, -10, 10, 30, 50]
y_pos_km = y_values[2]

# cmap = 'coolwarm'
cmap = 'gist_rainbow_r'
# cmap = 'jet'

fig, axes = plt.subplots(5, 1, figsize=(8, 14), dpi=300)
for i, y_pos_km in enumerate(y_values):
    plot_vertical_slice_pyvista(vs_grid, field_name, position_m=y_pos_km*1e3, direction='y',
                                ax=axes[i], vmin=vmin_global, vmax=vmax_global,
                                cmap=cmap, add_colorbar=True, levels=256)
    for depth in depths:
        axes[i].axhline(y=-depth, color='k', linestyle='--', linewidth=1)
    axes[i].set_title(f'Mesh-Projected Model (Vertical Slice at Y = {y_pos_km} km)',
                    fontsize=14, fontweight='bold')
    axes[i].set_xlim([-160, 140])
    axes[i].set_ylim([-90, 0])
    # axes[i].set_xlim([-1000, 1000])
    # axes[i].set_ylim([-400, 0])


In [ ]:
# %%
# ========================================
# COMPARISON: Original CSV Model vs Mesh-Projected Model (Vertical Slices)
# ========================================
pltcol = 'vs'
vmin_global = 2.5
vmax_global = 6

# y_pos_km = 0  # Position for vertical slice
# y_pos_km = np.round(y_line.mean())/1e3
y_values = [-50, -25, 0, 25, 50]
y_pos_km = y_values[3]

# cmap = 'coolwarm'
cmap = 'gist_rainbow_r'
# cmap = 'jet'

fig, axes = plt.subplots(2, 1, figsize=(9, 6), dpi=300)

print("=" * 60)
print("VERTICAL SLICE COMPARISON")
print("=" * 60)

# Top panel: Original CSV model
print(f"\nPlotting original CSV model at Y = {y_pos_km} km...")
plot_vertical_slice(vel3d, pltcol, ax=axes[0], position_km=y_pos_km, direction='y',
                   vmin=vmin_global, vmax=vmax_global, resolution=250,
                   add_contour=False, add_colorbar=True, cmap=cmap,
                   no_interpolation=True, marker='s', marker_size=65,
                   xcol='x', ycol='y', zcol='z',
                   xlim=[-160, 140], ylim=[-90, 0])

for depth in depths:
    axes[0].axhline(y=-depth, color='k', linestyle='--', linewidth=1)
axes[0].set_title(f'Original CSV Model (Vertical Slice at Y = {y_pos_km} km)',
                  fontsize=14, fontweight='bold')
axes[0].set_xlim([-160, 140])
axes[0].set_ylim([-90, 0])

# Bottom panel: Mesh-projected model
print(f"\nPlotting mesh-projected model at Y = {y_pos_km} km...")
plot_vertical_slice_pyvista(vs_grid, field_name, position_m=y_pos_km*1e3, direction='y',
                            ax=axes[1], vmin=vmin_global, vmax=vmax_global,
                            cmap=cmap, add_colorbar=True, levels=256)
for depth in depths:
    axes[1].axhline(y=-depth, color='k', linestyle='--', linewidth=1)
axes[1].set_title(f'Mesh-Projected Model (Vertical Slice at Y = {y_pos_km} km)',
                  fontsize=14, fontweight='bold')
axes[1].set_xlim([-160, 140])
axes[1].set_ylim([-90, 0])
# axes[1].set_xlim([-500, 500])
# axes[1].set_ylim([-100, 0])

fig.suptitle('Comparison: Original vs Mesh-Projected Velocity Model (Vertical Slices)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as mtri

def plot_horizontal_slice_geo(grid, field_name, depth_m, lon0, lat0, rot, x0, y0,
                               ax=None, vmin=None, vmax=None, cmap='gist_rainbow_r',
                               add_colorbar=True, levels=256, 
                               lon_range=None, lat_range=None,
                               add_features=True, gridlines=True):
    """
    Plot horizontal slice in geographic coordinates with coastlines
    
    Parameters:
    -----------
    grid : pv.UnstructuredGrid
        PyVista grid with velocity data
    field_name : str
        Name of the field to plot
    depth_m : float
        Depth in meters (negative value)
    lon0, lat0 : float
        Origin for coordinate transformation
    rot : float
        Rotation angle in degrees
    x0, y0 : float
        Offset in meters
    ax : cartopy axes
        Axes with PlateCarree projection (if None, creates new figure)
    vmin, vmax : float
        Color limits
    cmap : str
        Colormap name
    add_colorbar : bool
        Whether to add colorbar
    levels : int
        Number of contour levels
    lon_range : tuple (lon_min, lon_max)
        Longitude range to display (default None, auto)
    lat_range : tuple (lat_min, lat_max)
        Latitude range to display (default None, auto)
    add_features : bool
        Whether to add land/ocean/coastline features
    gridlines : bool
        Whether to add gridlines with labels
        
    Returns:
    --------
    CS : contourf object
    cbar : colorbar object or None
    """
    # Slice the grid
    normal = [0., 0., 1.]
    origin = [0., 0., depth_m]
    slice_mesh = grid.slice(normal=normal, origin=origin)
    
    if len(slice_mesh.points) == 0:
        print(f"No data at depth {depth_m/1e3:.1f} km")
        return None, None
    
    if field_name == 'shear modulus':
        unit = 'GPa'
    elif field_name == 'shear velocity Vs':
        unit = 'km/s'
    elif field_name == 'density':    
        unit = 'kg m/s^2'

    # Extract x, y coordinates and values
    x = slice_mesh.points[:, 0]
    y = slice_mesh.points[:, 1]
    values = slice_mesh[field_name]
    
    # Convert back to lon/lat
    x_unshifted = x + x0
    y_unshifted = y + y0
    lon, lat = utp.ckm2LLd(x_unshifted, y_unshifted, lon0, lat0, -rot)
    
    # Create axes if not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 10), dpi=300, 
                               subplot_kw={'projection': ccrs.PlateCarree()})
        standalone = True
    else:
        standalone = False
    
    # Add map features
    if add_features:
        # ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
        # ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.2)
        ax.add_feature(cfeature.COASTLINE, linewidth=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')
    
    # Set color limits
    if vmin is None:
        vmin = values.min()
    if vmax is None:
        vmax = values.max()
    
    # Create triangulation and plot
    triang = mtri.Triangulation(lon, lat)
    contour_levels = np.linspace(vmin, vmax, levels)
    CS = ax.tricontourf(triang, values, levels=contour_levels, cmap=cmap, 
                        vmin=vmin, vmax=vmax, extend='both', 
                        transform=ccrs.PlateCarree())
    
    # Add colorbar
    cbar = None
    if add_colorbar:
        cbar = plt.colorbar(CS, ax=ax, orientation='vertical', pad=0.02, shrink=0.8)
        cbar.set_label(f'{field_name} ({unit})', fontsize=12)
        cbar.set_ticks(np.linspace(vmin, vmax, 6))
    
    # Set extent
    if lon_range is not None or lat_range is not None:
        lon_min = lon_range[0] if lon_range else lon.min()
        lon_max = lon_range[1] if lon_range else lon.max()
        lat_min = lat_range[0] if lat_range else lat.min()
        lat_max = lat_range[1] if lat_range else lat.max()
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    
    # Add gridlines
    if gridlines:
        gl = ax.gridlines(draw_labels=True, alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
    
    ax.set_title(f'Horizontal Slice at Depth = {-depth_m/1e3:.1f} km\n({len(values)} points)', 
                 fontsize=14, fontweight='bold')
    
    if standalone:
        plt.tight_layout()
        plt.show()
    
    print(f"{field_name} range: {values.min():.2f} - {values.max():.2f} {unit}")
    
    return CS, cbar

In [ ]:
# %%
# Comparison: Original CSV vs Mesh-Projected in Geographic Coordinates
fig = plt.figure(figsize=(19, 9), dpi=300)
depths = [9, 20, 35, 50]

pltcol = 'vs'
vmin_global = 2.5
vmax_global = 6
# cmap = 'jet'
cmap='gist_rainbow_r'

region1=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

for i, depth in enumerate(depths):
    # Top row: Original CSV model using existing function
    ax1 = fig.add_subplot(2, 4, i+1, projection=ccrs.PlateCarree())
    
    # Use existing plot_horizontal_slice function with lon/lat coordinates
    plot_horizontal_slice(vel3d, pltcol, depth_km=depth, ax=ax1,
                         vmin=vmin_global, vmax=vmax_global, cmap=cmap,
                         no_interpolation=True, marker='s', marker_size=70,
                         xcol='lon', ycol='lat', zcol='z', xlim=region1[0:2], ylim=region1[2:4])

    ax1.plot(lon_origin, lat_origin, 'r*', markersize=10, transform=ccrs.PlateCarree(), label=f'Origin (0, 0)')

    # Add coastline features on top
    # ax1.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3, zorder=0)
    # ax1.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.2, zorder=0)
    ax1.add_feature(cfeature.COASTLINE, linewidth=1, zorder=10)
    ax1.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', zorder=10)

    # Add gridlines
    gl = ax1.gridlines(draw_labels=True, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    
    ax1.set_title(f'Original CSV (no interp)\nDepth = {depth} km', fontsize=12, fontweight='bold')
    
    # Bottom row: Mesh-projected model
    ax2 = fig.add_subplot(2, 4, i+5, projection=ccrs.PlateCarree())
    plot_horizontal_slice_geo(vs_grid, 'shear velocity Vs', depth_m=-depth*1e3,
                              lon0=lon0, lat0=lat0, rot=rot, x0=x0, y0=y0,
                              ax=ax2, vmin=vmin_global, vmax=vmax_global, cmap=cmap,
                              lon_range=region1[0:2], lat_range=region1[2:4])
    ax2.plot(lon_origin, lat_origin, 'r*', markersize=10, transform=ccrs.PlateCarree(), label=f'Origin (0, 0)')
    
    # y_values = [0, np.round(y_line.mean())]
    y_values = [-50, -25, 0, 25, 50]
    for y_val in y_values:
        # Create arrays of x and y coordinates
        x_coords = x_range
        y_coords = np.full_like(x_range, y_val*1e3)
        
        # Reverse the offset (add back x0, y0)
        x_unshifted = x_coords + x0
        y_unshifted = y_coords + y0
        
        # Reverse the rotation and convert back to lon/lat
        # Note: use -rot to reverse the rotation
        lon_line, lat_line = utp.ckm2LLd(x_unshifted, y_unshifted, lon0, lat0, -rot)
        
        # Now you can plot these lines
        ax1.plot(lon_line, lat_line, 'k--', linewidth=1, label=f'y={y_val:.0f} km')
        ax2.plot(lon_line, lat_line, 'k--', linewidth=1, label=f'y={y_val:.0f} km')

    ax2.set_title(f'Mesh-Projected\nDepth = {depth} km', fontsize=12, fontweight='bold')

fig.suptitle('Comparison: Original CSV vs Mesh-Projected (Geographic Coordinates with Coastlines)',
             fontsize=16, fontweight='bold')
plt.tight_layout()

In [ ]:
# Load the mesh-projected shear modulus model (.xdmf file)
print(f"Loading {vs_file}...")
mu_grid = load_xdmf_as_pyvista(xdmf_dir + mu_file)

# Check the field name
field_name_mu = mu_grid.array_names[0] if len(mu_grid.array_names) > 0 else 'shear modulus'
print(f"\nUsing field: '{field_name_mu}'")

# Get global min/max for consistent colorbar
mu_values = mu_grid[field_name_mu]
mu_min_global = mu_values.min()
mu_max_global = mu_values.max()
print(f"Global Vs range: {mu_min_global:.2f} - {mu_max_global:.2f} GPa\n")

In [ ]:
# Mesh-Projected shear modulus in Geographic Coordinates
fig = plt.figure(figsize=(8.5, 7), dpi=300)
depths = [9, 20, 35, 50]

cmap='gist_rainbow_r'

region1=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

for i, depth in enumerate(depths):
    # Mesh-projected model
    ax = fig.add_subplot(2, 2, i+1, projection=ccrs.PlateCarree())
    plot_horizontal_slice_geo(mu_grid, field_name_mu, depth_m=-depth*1e3,
                              lon0=lon0, lat0=lat0, rot=rot, x0=x0, y0=y0,
                              ax=ax, 
                              vmin=mu_min_global, vmax=mu_max_global, cmap=cmap,
                              lon_range=region1[0:2], lat_range=region1[2:4])
    # y_values = [0, np.round(y_line.mean())]
    y_values = [-50, -25, 0, 25, 50]
    for y_val in y_values:
        # Create arrays of x and y coordinates
        x_coords = x_range
        y_coords = np.full_like(x_range, y_val*1e3)
        
        # Reverse the offset (add back x0, y0)
        x_unshifted = x_coords + x0
        y_unshifted = y_coords + y0
        
        # Reverse the rotation and convert back to lon/lat
        # Note: use -rot to reverse the rotation
        lon_line, lat_line = utp.ckm2LLd(x_unshifted, y_unshifted, lon0, lat0, -rot)
        
        # Now you can plot these lines
        ax.plot(lon_line, lat_line, 'k--', linewidth=1, label=f'y={y_val:.0f} km')

    ax.set_title(f'Depth = {depth} km', fontsize=12, fontweight='bold')

fig.suptitle('Mesh-Projected shear modulus', fontsize=16, fontweight='bold')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 6), dpi=300)
axes = axes.flatten()  # Flatten to 1D array for easier indexing

# pltcol = 'vs'
# vmin_global = 2.5
# vmax_global = 6

# y_pos_km = 0  # Position for vertical slice
# y_pos_km = np.round(y_line.mean())/1e3

# y_values = [-50, -25, 0, 25, 50]
y_values = [-50, -25, -10, 10, 30, 50]

# cmap = 'coolwarm'
cmap = 'gist_rainbow_r'
# cmap = 'jet'
    
for i, y_val in enumerate(y_values):
    plot_vertical_slice_pyvista(mu_grid, field_name_mu, position_m=y_val*1e3, direction='y',
                                ax=axes[i], vmin=mu_min_global, vmax=mu_max_global,
                                cmap=cmap, add_colorbar=True, levels=256)
    for depth in depths:
        axes[i].axhline(y=-depth, color='k', linestyle='--', linewidth=1)

    axes[i].set_title(f'Y = {y_val} km',
                    fontsize=14, fontweight='bold')
    axes[i].set_xlim([-160, 140])
    axes[i].set_ylim([-80, 0])

fig.suptitle('Mesh-Projected shear modulus (Vertical Slices)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()